Download heart disease dataset heart.csv in [Exercise](https://github.com/codebasics/py/tree/master/ML/19_Bagging/Exercise) folder and do following, (credits of dataset:  https://www.kaggle.com/fedesoriano/heart-failure-prediction)

1. Load heart disease dataset in pandas dataframe
1. Remove outliers using Z score. Usual guideline is to remove anything that has Z score > 3 formula or Z score < -3
1. Convert text columns to numbers using label encoding and one hot encoding
1. Apply scaling
1. Build a classification model using support vector machine. Use standalone model as well as Bagging model and check if you see any difference in the performance.
1. Now use decision tree classifier. Use standalone model as well as Bagging and check if you notice any difference in performance
1. Comparing performance of svm and decision tree classifier figure out where it makes most sense to use bagging and why. Use internet to figure out in what conditions bagging works the best.






 

In [40]:
import pandas as pd
df=pd.read_csv("heart.csv")

In [41]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [42]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [43]:
df.isnull().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [44]:
df.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')

In [45]:
X=df.drop(["HeartDisease"],axis="columns")
y=df["HeartDisease"]

In [46]:
X_encoded=pd.get_dummies(X,drop_first=True)

In [47]:
X_encoded.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,True,False,True,False,True,False,False,False,True


In [48]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
z_scores=scaler.fit_transform(X_encoded)

In [49]:
import numpy as np

In [50]:
threshold=3
mask=(np.abs(z_scores) < threshold).all(axis=1)
X_filtered=X_encoded[mask]
y_filtered=y[mask]

In [51]:
from sklearn.preprocessing import StandardScaler

scaler1=StandardScaler()
X_scaled=scaler1.fit_transform(X_filtered)

In [52]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingClassifier

# Step 1: Tune SVM on full training data
param_grids = {
    "C": list(range(1, 10)),      # smaller range for speed
    "kernel": ['linear', 'rbf'],  # fewer kernels for speed
    "gamma": ['auto']
}

svm_clf = GridSearchCV(SVC(), param_grid=param_grids, cv=5)
svm_clf.fit(X_scaled, y_filtered)  # fit once
print("Best SVM parameters:", svm_clf.best_params_)
print("Best CV score:", svm_clf.best_score_)

# Step 2: Use the best SVM as Bagging estimator
best_svm = svm_clf.best_estimator_

bag_model = BaggingClassifier(
    estimator=best_svm,
    n_estimators=30,     # reduced from 100 for speed
    max_samples=0.8,
    oob_score=True,
    random_state=10,
    n_jobs=-1            # use all cores for speed
)

bag_model.fit(X_scaled, y_filtered)
print("Bagging OOB score:", bag_model.oob_score_)


Best SVM parameters: {'C': 1, 'gamma': 'auto', 'kernel': 'rbf'}
Best CV score: 0.8477055383556932
Bagging OOB score: 0.8782201405152225


In [53]:
from sklearn.tree import DecisionTreeClassifier
model=DecisionTreeClassifier()
param_grids={
    "criterion":['gini', 'entropy', 'log_loss'],
    "splitter":['best', 'random']
}

clf1=GridSearchCV(model,param_grid=param_grids)
scores1=cross_val_score(clf1,X_filtered,y_filtered,cv=5)
scores1.mean()

np.float64(0.7786033711730307)

In [56]:
from sklearn.model_selection import cross_val_score

In [57]:
from sklearn.ensemble import BaggingClassifier

bag_model=BaggingClassifier(
    estimator=clf1,
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=10
)

scores=cross_val_score(bag_model,X_filtered,y_filtered,cv=5)
scores.mean()

np.float64(0.8277812177502579)